- python 3.13.9


In [2]:
import joblib
import pandas as pd
import numpy as np
import sys

# utils, submit에서 필요한 함수 로드
import importlib
import utils as ut
importlib.reload(ut)

import submit as sb
importlib.reload(sb)

import save_log as sv
importlib.reload(sv)


import elo
importlib.reload(elo)

<module 'elo' from 'c:\\Users\\User\\Desktop\\statiz_prediction\\elo.py'>

In [3]:
loaded_model_elo = joblib.load("logit_model_elo_full.pkl")
feature_cols_elo = joblib.load("feature_cols_elo_full.pkl")
loaded_model_elo_sp = joblib.load("logit_model_elo_sp.pkl")
feature_cols_elo_sp = joblib.load("feature_cols_elo_sp.pkl")
print("ELO 모델 로드 완료")


ELO 모델 로드 완료


### API에서 오늘 경기 데이터 불러오기

### Given lineup
- 날짜 변경 필수

In [4]:
# 예측 데이터 구축
today = pd.Timestamp.now(tz="Asia/Seoul").normalize().tz_localize(None)
pred_result = ut.build_prediction_data_for_date(  
    date_str=today.strftime("%Y-%m-%d"),  # 직접 지정 시 예: "2026-06-01"
    # date_str="2026-06-02",
    sleep_sec=0.12,
    verbose=True
)

pred_df = pred_result["pred_df"]


=== 1. collect_games ===
[games] 2026-08-25: 5 rows
games_df.shape = (5, 21)


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20260616,2026-08-25 09:30:00,2026,5002,11001,1001,16404,16450,톨허스트,테일러,...,None,0,1,18:30:00,0,0,0,0,0,0
1,20260617,2026-08-25 09:30:00,2026,10001,1001,1003,13941,16666,알칸타라,보스,...,None,0,1,18:30:00,0,0,0,0,0,0
2,20260618,2026-08-25 09:30:00,2026,9002,7002,2001,16440,16422,김민준,화이트,...,None,0,1,18:30:00,0,0,0,0,0,0
3,20260619,2026-08-25 09:30:00,2026,12001,6002,2002,14600,16288,소형준,최민석,...,None,0,1,18:30:00,0,0,0,0,0,0
4,20260620,2026-08-25 09:30:00,2026,2002,3001,6001,16148,16433,시라카와,비슬리,...,None,0,1,18:30:00,0,0,0,0,0,0


=== 2. filter target games ===
games_pred.shape = (5, 21)


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20260616,2026-08-25 09:30:00,2026,5002,11001,1001,16404,16450,톨허스트,테일러,...,None,0,1,18:30:00,0,0,0,0,0,0
1,20260617,2026-08-25 09:30:00,2026,10001,1001,1003,13941,16666,알칸타라,보스,...,None,0,1,18:30:00,0,0,0,0,0,0
2,20260618,2026-08-25 09:30:00,2026,9002,7002,2001,16440,16422,김민준,화이트,...,None,0,1,18:30:00,0,0,0,0,0,0
3,20260619,2026-08-25 09:30:00,2026,12001,6002,2002,14600,16288,소형준,최민석,...,None,0,1,18:30:00,0,0,0,0,0,0
4,20260620,2026-08-25 09:30:00,2026,2002,3001,6001,16148,16433,시라카와,비슬리,...,None,0,1,18:30:00,0,0,0,0,0,0


=== 3. collect_lineups ===
[lineup] 20260616: 2 rows
[lineup] 20260617: 2 rows
[lineup] 20260618: 2 rows
[lineup] 20260619: 2 rows
[lineup] 20260620: 2 rows
lineup_df.shape = (10, 13)
=== 4. collect_rosters ===
[roster] 2026-08-25 team=1001: 30 rows
[roster] 2026-08-25 team=2002: 30 rows
[roster] 2026-08-25 team=3001: 30 rows
[roster] 2026-08-25 team=5002: 30 rows
[roster] 2026-08-25 team=6002: 30 rows
[roster] 2026-08-25 team=7002: 30 rows
[roster] 2026-08-25 team=9002: 30 rows
[roster] 2026-08-25 team=10001: 30 rows
[roster] 2026-08-25 team=11001: 30 rows
[roster] 2026-08-25 team=12001: 30 rows
roster_df.shape = (300, 5)
n_unique_player_ids = 300
=== 5. collect_player_season ===
[playerSeason] 10892: OK
[playerSeason] 10527: OK
[playerSeason] 10232: OK
[playerSeason] 14113: OK
[playerSeason] 14116: OK
[playerSeason] 14618: OK
[playerSeason] 12936: OK
[playerSeason] 14798: OK
[playerSeason] 11344: OK
[playerSeason] 15034: OK
[playerSeason] 15035: OK
[playerSeason] 11336: OK
[playerSea

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_sp_KBB,away_sp_QS,away_sp_GS,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff
0,20260616,2026-08-25 09:30:00,2026,5002,11001,1001,16404,16450,톨허스트,테일러,...,1.8478,7,21,-0.35,-0.63288,-0.07,0.754,-1.372,0.168,1.4425
1,20260617,2026-08-25 09:30:00,2026,10001,1001,1003,13941,16666,알칸타라,보스,...,3.8333,0,4,-3.56,1.36954,-0.84,-4.555,-2.160,0.705,3.6667
2,20260618,2026-08-25 09:30:00,2026,9002,7002,2001,16440,16422,김민준,화이트,...,3.6000,9,17,0.45,1.05653,0.01,-0.446,2.049,0.098,-2.0167
3,20260619,2026-08-25 09:30:00,2026,12001,6002,2002,14600,16288,소형준,최민석,...,2.0392,12,21,0.67,-0.37267,0.00,-0.819,-2.274,0.209,2.2549
4,20260620,2026-08-25 09:30:00,2026,2002,3001,6001,16148,16433,시라카와,비슬리,...,3.5588,9,21,-0.28,1.72725,0.04,-2.420,2.831,0.180,-2.2685


In [ ]:
pred_df.isna().sum().sum()  # 정상은 10

np.int64(10)

### 선발투수 결측치 보간


In [5]:
season_pit_df = pd.read_csv('seaon_pit_df.csv')

pred_df = ut.fill_and_recompute_sp_features(
    pred_df=pred_df,
    season_pit_df=season_pit_df,
    target_year=2026
)

In [11]:
pred_df

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_sp_KBB,away_sp_QS,away_sp_GS,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff
0,20260356,2026-06-19 09:30:00,2026,5002,6002,1001,14134,15143,이정용,벤자민,...,3.1250,6,10,3.56,1.92697,0.59,-3.129,1.202,0.376,-1.8393
1,20260357,2026-06-19 09:30:00,2026,10001,3001,1003,13941,15045,알칸타라,이민석,...,1.1538,0,3,-3.61,-1.59642,-0.53,2.574,-3.472,0.435,5.1795
2,20260358,2026-06-19 09:30:00,2026,12001,2002,2002,14581,16088,오원석,네일,...,4.1333,8,14,1.69,0.03162,0.35,1.239,0.614,0.379,-0.5708
3,20260359,2026-06-19 09:30:00,2026,7002,1001,4003,16572,15531,박준영,후라도,...,3.6667,10,13,1.17,2.87591,-0.02,0.713,1.354,1.747,-1.4167
4,20260360,2026-06-19 09:30:00,2026,11001,9002,8005,16454,16446,토다,베니지아노,...,2.0385,1,13,-0.22,-0.48890,0.04,-0.662,-0.336,-0.198,0.0070


In [6]:
pred_df.isna().sum().sum()  # Encoding-normalized comment.

np.int64(12)

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,home_sp_imputed,away_sp_imputed,home_win_proba
0,20260111,2026-04-23 09:30:00,2026,5002,7002,1001,14134,16107,이정용,황준서,...,1.76,2.69464,0.33,-7.232,-0.643,0.000,-1.4167,0,0,0.254640
1,20260112,2026-04-23 09:30:00,2026,10001,11001,1003,16127,11353,김연주,구창모,...,3.80,2.44236,0.75,-1.168,1.131,0.936,-0.7550,1,0,0.274754
2,20260113,2026-04-23 09:30:00,2026,12001,2002,2002,14600,14788,소형준,이의리,...,-2.74,-3.18603,-0.84,-1.537,-6.266,-1.158,4.1667,0,0,0.651515
3,20260114,2026-04-23 09:30:00,2026,1001,9002,7003,13574,16324,오러클린,화이트,...,0.40,0.11089,-0.19,0.036,-0.417,-0.435,0.2000,0,0,0.456136
4,20260115,2026-04-23 09:30:00,2026,3001,6002,8001,16434,16299,로드리게스,잭로그,...,1.45,1.61294,0.36,-0.535,3.241,0.535,-2.5385,0,0,0.421094


### Add Recent Flow Features for Reduced Core

In [5]:
# 오늘날짜에서 14일전부터 수집
pred_df_core = ut.add_recent_flow_features_for_prediction(pred_df, sleep_sec=0.15)

[games] 2026-08-11: 5 rows
[games] 2026-08-12: 5 rows
[games] 2026-08-13: 5 rows
[games] 2026-08-14: 5 rows
[games] 2026-08-15: 5 rows
[games] 2026-08-16: 5 rows
[games] 2026-08-17: 0 rows
[games] 2026-08-18: 5 rows
[games] 2026-08-19: 5 rows
[games] 2026-08-20: 5 rows
[games] 2026-08-21: 5 rows
[games] 2026-08-22: 5 rows
[games] 2026-08-23: 5 rows
[games] 2026-08-24: 0 rows
[recent] lookback_start = 2026-08-11 | completed_games_by_team = {'10001': 12, '1001': 11, '11001': 11, '12001': 11, '2002': 12, '3001': 10, '5002': 11, '6002': 11, '7002': 10, '9002': 11}
missing feature cols after recent merge: ['lineup_OPS_diff', 'lineup_OBP_diff']


c:\Users\User\Desktop\statiz_prediction\utils.py:220: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()


,game_id,game_date,home_team_code,away_team_code,recent_10_win_rate_diff,recent_5_run_diff_avg_diff,recent_5_runs_against_avg_diff,rest_days_diff,prev_game_win_diff
0,20260616,2026-08-25 09:30:00,5002,11001,0.000000,-1.0,2.0,1.0,0.0
1,20260617,2026-08-25 09:30:00,10001,1001,-0.200000,-6.4,3.0,-1.0,0.0
2,20260618,2026-08-25 09:30:00,9002,7002,0.377778,0.8,-2.6,0.0,-1.0
3,20260619,2026-08-25 09:30:00,12001,6002,-0.100000,1.6,-1.0,0.0,0.0
4,20260620,2026-08-25 09:30:00,2002,3001,0.033333,-1.2,-1.6,-2.0,-1.0


### Add elo features
- daily update needed

In [6]:
sys.path.append(r"C:\Users\User\Desktop\statiz_prediction")
games_with_elo, updated_snapshot = elo.update_team_elo_snapshot_2026() # print 생략됨

pred_df_elo = elo.merge_elo(pred_df_core)

UPDATE_TARGET_DATE: 2026-08-25
[games] 2026-08-24: 0 rows
No completed games found from 2026-08-24 to 2026-08-24


,game_id,home_team_code,away_team_code,home_elo_pre,away_elo_pre,elo_diff,elo_exp_home
0,20260616,5002,11001,1476.583758,1473.497277,3.086480,0.540332
1,20260617,10001,1001,1424.112815,1551.486009,-127.373194,0.356794
2,20260618,9002,7002,1468.027164,1448.408671,19.618493,0.563860
3,20260619,12001,6002,1571.856102,1547.462451,24.393651,0.570608
4,20260620,2002,3001,1532.931637,1505.634117,27.297520,0.574699


missing elo rows: 0


In [9]:
pred_df_elo

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,recent_10_run_diff_avg_diff,rest_days_diff,prev_game_win_diff,home_elo_pre,latest_update_date_x,away_elo_pre,latest_update_date_y,elo_diff,elo_exp_home,home_win_proba
0,20260261,2026-05-28 09:30:00,2026,6002,12001,1001,13061,11308,곽빈,고영표,...,0.700000,0.0,-1.0,1485.214596,2026-05-27,1513.726272,2026-05-27,-28.511676,0.494946,0.577884
1,20260262,2026-05-28 09:30:00,2026,10001,2002,1003,16339,15063,로젠버그,황동하,...,-2.500000,0.0,-1.0,1454.063089,2026-05-27,1541.467278,2026-05-27,-87.404190,0.411147,0.702044
2,20260263,2026-05-28 09:30:00,2026,9002,1001,2001,16637,11379,긴지로,최원태,...,-2.900000,0.0,-1.0,1453.414287,2026-05-27,1570.990829,2026-05-27,-117.576542,0.369838,0.293886
3,20260264,2026-05-28 09:30:00,2026,3001,5002,8001,14792,14134,김진욱,이정용,...,1.000000,0.0,-1.0,1458.337861,2026-05-27,1556.697400,2026-05-27,-98.359538,0.395969,0.584173
4,20260265,2026-05-28 09:30:00,2026,11001,7002,8005,14593,16424,김태경,왕옌청,...,-2.555556,0.0,0.0,1453.967540,2026-05-27,1512.120848,2026-05-27,-58.153308,0.452433,0.489174


### logit_elo_full

In [7]:
# feature 순서 맞추기
missing_elo_cols = [c for c in feature_cols_elo if c not in pred_df_elo.columns]
if missing_elo_cols:
    raise KeyError(f"elo full 모델 입력 컬럼이 pred_df_elo에 없습니다: {missing_elo_cols}")

X_pred = pred_df_elo[feature_cols_elo].copy()
X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_elo["home_win_proba"] = loaded_model_elo.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df_elo[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260326,2026-06-12 09:30:00,5002,3001,0.597616
1,20260327,2026-06-12 09:30:00,10001,7002,0.385976
2,20260328,2026-06-12 09:30:00,12001,11001,0.547897
3,20260329,2026-06-12 09:30:00,1001,9002,0.601734
4,20260330,2026-06-12 09:30:00,2002,6002,0.454874


In [8]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260326           59.76
1  20260327           38.60
2  20260328           54.79
3  20260329           60.17
4  20260330           45.49


### logit_elo_sp

In [7]:
# feature 순서 맞추기
missing_elo_cols = [c for c in feature_cols_elo_sp if c not in pred_df_elo.columns]
if missing_elo_cols:
    raise KeyError(f"elo full 모델 입력 컬럼이 pred_df_elo에 없습니다: {missing_elo_cols}")

X_pred = pred_df_elo[feature_cols_elo_sp].copy()
X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_elo["home_win_proba"] = loaded_model_elo_sp.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df_elo[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260616,2026-08-25 09:30:00,5002,11001,0.598670
1,20260617,2026-08-25 09:30:00,10001,1001,0.393820
2,20260618,2026-08-25 09:30:00,9002,7002,0.498618
3,20260619,2026-08-25 09:30:00,12001,6002,0.466636
4,20260620,2026-08-25 09:30:00,2002,3001,0.339073


In [8]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260616           59.87
1  20260617           39.38
2  20260618           49.86
3  20260619           46.66
4  20260620           33.91


### Logistic_baseline_elo_sp_selected


In [7]:
# modeling_5th_experiment에서 accuracy 기준으로 채택한 모델
# - feature set: elo_sp
# - model: LogisticRegression(C=0.1, penalty='l2')

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def _make_team_game_log_selected(game_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["game_id", "game_date", "season"]

    home = game_df[base_cols + ["home_team_code", "away_team_code", "home_score", "away_score", "elo_result_home"]].copy()
    home = home.rename(columns={
        "home_team_code": "team_code",
        "away_team_code": "opponent_team_code",
        "home_score": "runs_for",
        "away_score": "runs_against",
        "elo_result_home": "result",
    })
    home["is_home"] = 1

    away = game_df[base_cols + ["away_team_code", "home_team_code", "away_score", "home_score", "elo_result_home"]].copy()
    away = away.rename(columns={
        "away_team_code": "team_code",
        "home_team_code": "opponent_team_code",
        "away_score": "runs_for",
        "home_score": "runs_against",
    })
    away["result"] = 1.0 - pd.to_numeric(away["elo_result_home"], errors="coerce")
    away = away.drop(columns=["elo_result_home"])
    away["is_home"] = 0

    team_games = pd.concat([home, away], ignore_index=True)
    team_games["game_date"] = pd.to_datetime(team_games["game_date"], errors="coerce")
    team_games["team_code"] = team_games["team_code"].astype(str)
    team_games["opponent_team_code"] = team_games["opponent_team_code"].astype(str)
    team_games["runs_for"] = pd.to_numeric(team_games["runs_for"], errors="coerce")
    team_games["runs_against"] = pd.to_numeric(team_games["runs_against"], errors="coerce")
    team_games["result"] = pd.to_numeric(team_games["result"], errors="coerce")
    team_games["run_diff"] = team_games["runs_for"] - team_games["runs_against"]
    return team_games.sort_values(["team_code", "game_date", "game_id"]).reset_index(drop=True)


def _build_recent_features_selected(game_df: pd.DataFrame, windows=(5, 10)) -> pd.DataFrame:
    team_recent = _make_team_game_log_selected(game_df)
    grouped = team_recent.groupby(["team_code", "season"], group_keys=False)

    team_recent["prev_game_date"] = grouped["game_date"].shift(1)
    team_recent["rest_days"] = (team_recent["game_date"].dt.normalize() - team_recent["prev_game_date"].dt.normalize()).dt.days - 1
    team_recent["rest_days"] = team_recent["rest_days"].clip(lower=0)
    team_recent["prev_game_result"] = grouped["result"].shift(1)

    rolling_targets = {
        "result": "result_avg",
        "runs_for": "runs_for_avg",
        "runs_against": "runs_against_avg",
        "run_diff": "run_diff_avg",
    }
    for window in windows:
        for src_col, out_name in rolling_targets.items():
            out_col = f"recent_{window}_{out_name}"
            team_recent[out_col] = grouped[src_col].transform(
                lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()
            )

    recent_feature_cols = ["game_id", "team_code", "rest_days", "prev_game_result"]
    recent_feature_cols += [c for c in team_recent.columns if c.startswith("recent_")]
    recent_features = team_recent[recent_feature_cols].copy()

    home_recent = recent_features.rename(
        columns={c: f"home_{c}" for c in recent_feature_cols if c not in ["game_id", "team_code"]}
    ).rename(columns={"team_code": "home_team_code"})
    away_recent = recent_features.rename(
        columns={c: f"away_{c}" for c in recent_feature_cols if c not in ["game_id", "team_code"]}
    ).rename(columns={"team_code": "away_team_code"})

    out = game_df.copy()
    out["home_team_code"] = out["home_team_code"].astype(str)
    out["away_team_code"] = out["away_team_code"].astype(str)
    out = out.merge(home_recent, on=["game_id", "home_team_code"], how="left")
    out = out.merge(away_recent, on=["game_id", "away_team_code"], how="left")

    diff_pairs = ["rest_days", "prev_game_result"] + [c for c in team_recent.columns if c.startswith("recent_")]
    for col in diff_pairs:
        out[f"{col}_diff"] = out[f"home_{col}"] - out[f"away_{col}"]
    return out


def _expected_home_score_selected(home_elo, away_elo, home_field_adv=25.0):
    return 1.0 / (1.0 + 10 ** (((away_elo) - (home_elo + home_field_adv)) / 400.0))


def _regress_to_mean_selected(old_elo, alpha=0.65, base_elo=1500.0):
    return base_elo + alpha * (old_elo - base_elo)


def _build_elo_features_selected(game_df, alpha=0.65, home_field_adv=25.0, base_elo=1500.0, k_factor=20.0):
    elo_df = game_df.copy().sort_values(["game_date", "game_id"]).reset_index(drop=True)
    elo_df["home_team_code"] = elo_df["home_team_code"].astype(str)
    elo_df["away_team_code"] = elo_df["away_team_code"].astype(str)

    ratings = {}
    last_seen_season = {}
    home_elo_pre_list = []
    away_elo_pre_list = []
    elo_exp_home_list = []
    home_elo_post_list = []
    away_elo_post_list = []

    for row in elo_df.itertuples(index=False):
        season = row.season
        home_team = row.home_team_code
        away_team = row.away_team_code

        for team in [home_team, away_team]:
            if team not in ratings:
                ratings[team] = base_elo
                last_seen_season[team] = season
            elif last_seen_season[team] != season:
                ratings[team] = _regress_to_mean_selected(ratings[team], alpha=alpha, base_elo=base_elo)
                last_seen_season[team] = season

        result_home = row.elo_result_home
        home_elo_pre = ratings[home_team]
        away_elo_pre = ratings[away_team]
        elo_exp_home = _expected_home_score_selected(home_elo_pre, away_elo_pre, home_field_adv=home_field_adv)

        elo_delta = k_factor * (result_home - elo_exp_home)
        home_elo_post = home_elo_pre + elo_delta
        away_elo_post = away_elo_pre - elo_delta

        ratings[home_team] = home_elo_post
        ratings[away_team] = away_elo_post
        last_seen_season[home_team] = season
        last_seen_season[away_team] = season

        home_elo_pre_list.append(home_elo_pre)
        away_elo_pre_list.append(away_elo_pre)
        elo_exp_home_list.append(elo_exp_home)
        home_elo_post_list.append(home_elo_post)
        away_elo_post_list.append(away_elo_post)

    elo_df["home_elo_pre"] = home_elo_pre_list
    elo_df["away_elo_pre"] = away_elo_pre_list
    elo_df["elo_diff"] = elo_df["home_elo_pre"] - elo_df["away_elo_pre"]
    elo_df["elo_exp_home"] = elo_exp_home_list
    elo_df["home_elo_post"] = home_elo_post_list
    elo_df["away_elo_post"] = away_elo_post_list
    return elo_df


train_base_df = pd.concat(
    [
        pd.read_csv("model_df_2023.csv"),
        pd.read_csv("model_df_2024.csv"),
        pd.read_csv("model_df_2025.csv"),
    ],
    ignore_index=True,
)
train_base_df["game_date"] = pd.to_datetime(train_base_df["game_date"], errors="coerce")
for col in ["home_score", "away_score", "home_win", "y_home_win", "season"]:
    if col in train_base_df.columns:
        train_base_df[col] = pd.to_numeric(train_base_df[col], errors="coerce")

train_base_df["elo_result_home"] = np.where(
    train_base_df["home_score"] > train_base_df["away_score"],
    1.0,
    np.where(train_base_df["home_score"] < train_base_df["away_score"], 0.0, 0.5),
)
train_base_df = train_base_df.sort_values(["game_date", "game_id"]).reset_index(drop=True)

train_model_df = _build_recent_features_selected(train_base_df, windows=(5, 10))
train_model_df = _build_elo_features_selected(train_model_df)

selected_recent_cols = [
    "recent_5_result_avg_diff",
    "recent_5_runs_for_avg_diff",
    "recent_5_runs_against_avg_diff",
    "recent_5_run_diff_avg_diff",
    "recent_10_result_avg_diff",
    "recent_10_runs_for_avg_diff",
    "recent_10_runs_against_avg_diff",
    "recent_10_run_diff_avg_diff",
    "rest_days_diff",
    "prev_game_result_diff",
]
selected_recent_cols = [c for c in selected_recent_cols if c in train_model_df.columns]
train_model_df[selected_recent_cols] = train_model_df[selected_recent_cols].fillna(0)

missing_train_cols = [c for c in feature_cols_elo_sp if c not in train_model_df.columns]
if missing_train_cols:
    raise KeyError(f"selected logistic baseline 학습 컬럼이 train_model_df에 없습니다: {missing_train_cols}")

train_ready = train_model_df.dropna(subset=feature_cols_elo_sp + ["y_home_win"]).copy()
X_train_selected = train_ready[feature_cols_elo_sp].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train_selected = train_ready["y_home_win"].astype(int)

selected_logistic_baseline_elo_sp = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=0.1, penalty="l2", solver="lbfgs", max_iter=2000, random_state=427)),
    ]
)
selected_logistic_baseline_elo_sp.fit(X_train_selected, y_train_selected)

# feature 순서 맞추기
missing_pred_cols = [c for c in feature_cols_elo_sp if c not in pred_df_elo.columns]
if missing_pred_cols:
    raise KeyError(f"selected logistic baseline 입력 컬럼이 pred_df_elo에 없습니다: {missing_pred_cols}")

X_pred_selected = pred_df_elo[feature_cols_elo_sp].copy()
X_pred_selected = X_pred_selected.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_elo["home_win_proba"] = selected_logistic_baseline_elo_sp.predict_proba(X_pred_selected)[:, 1]

# 결과 정리
result_df = pred_df_elo[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba",
    ]
].copy()
result_df["model_name"] = "logistic_baseline_elo_sp_selected"

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba,model_name
0,20260616,2026-08-25 09:30:00,5002,11001,0.550743,logistic_baseline_elo_sp_selected
1,20260617,2026-08-25 09:30:00,10001,1001,0.480435,logistic_baseline_elo_sp_selected
2,20260618,2026-08-25 09:30:00,9002,7002,0.483357,logistic_baseline_elo_sp_selected
3,20260619,2026-08-25 09:30:00,12001,6002,0.477724,logistic_baseline_elo_sp_selected
4,20260620,2026-08-25 09:30:00,2002,3001,0.391517,logistic_baseline_elo_sp_selected


In [8]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260616           55.07
1  20260617           48.04
2  20260618           48.34
3  20260619           47.77
4  20260620           39.15


### API Submit

In [9]:
# 예측 결과 로그 저장 at local

sv.append_prediction_log(
    pred_df=result_df,
    file_path="prediction_log.xlsx",
    sheet_name="Raw_Submission_Log",
    dedup=True,
    model_name="elo_sp_l2", # 모델명 기록 (elo_sp_l2, elo_full, elo_sp, core_full, core_sp, full, sp, non)
)

Saved log: prediction_log.xlsx
Total rows: 532


,game_id,game_date,home_team,away_team,home_team_code,away_team_code,home_win_percent,home_win_proba,model_name,result,correct
0,20260046,2026-04-08,두산,키움,6002,10001,88.72,0.887200,full,1,NaN
1,20260047,2026-04-08,SSG,한화,9002,7002,90.58,0.905800,full,0,NaN
2,20260048,2026-04-08,KIA,삼성,2002,1001,34.54,0.345400,full,1,NaN
3,20260049,2026-04-08,롯데,KT,3001,12001,8.49,0.084900,full,1,NaN
4,20260050,2026-04-08,NC,LG,11001,5002,12.38,0.123800,full,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
527,20260616,2026-08-25,LG,NC,5002,11001,55.07,0.550743,elo_sp_l2,NaN,NaN
528,20260617,2026-08-25,키움,삼성,10001,1001,48.04,0.480435,elo_sp_l2,NaN,NaN
529,20260618,2026-08-25,SSG,한화,9002,7002,48.34,0.483357,elo_sp_l2,NaN,NaN
530,20260619,2026-08-25,KT,두산,12001,6002,47.77,0.477724,elo_sp_l2,NaN,NaN


In [10]:
submit_result_df = sb.submit_predictions_df(
    submit_df,
    path="prediction/savePrediction",     
    sleep_sec=0.2,
    verbose=True
)

display(submit_result_df)

[SUBMIT OK] s_no=20260616, percent=55.07, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260617, percent=48.04, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260618, percent=48.34, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260619, percent=47.77, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260620, percent=39.15, result_cd=100, result_msg=저장 성공


,game_id,percent,result_cd,result_msg,raw_response
0,20260616,55.07,100,저장 성공,"{'percent': 55.07, 's_no': '20260616', 'code':..."
1,20260617,48.04,100,저장 성공,"{'percent': 48.04, 's_no': '20260617', 'code':..."
2,20260618,48.34,100,저장 성공,"{'percent': 48.34, 's_no': '20260618', 'code':..."
3,20260619,47.77,100,저장 성공,"{'percent': 47.77, 's_no': '20260619', 'code':..."
4,20260620,39.15,100,저장 성공,"{'percent': 39.15, 's_no': '20260620', 'code':..."
